In [ ]:
from pathlib import Path
import uproot
import numpy as np
from matplotlib import pyplot as plt
import mplhep as hep
from correctionlib import schemav2
import correctionlib

hep.style.use("CMS")

In [ ]:
plot_dir = Path("plots/tau32")
sf_dir = Path("SF")

plot_dir.mkdir(exist_ok=True, parents=True)
sf_dir.mkdir(exist_ok=True)

In [ ]:
LUMI_DICT = {
    "2022": 34652.1,
    "2023": 27776.5,
    "2024": 108960.0,
}
ylim_dict = {
    "2022": (1, 1e7),
    "2023": (1, 1e7),
}

In [ ]:
def binning(edges, content, var_name):
    return schemav2.Binning(
        nodetype="binning",
        input=var_name,
        edges=edges,
        content=list(content),
        flow="clamp",
    )


def get_corr(edges, val_nom, val_up, val_down, var_name, var_description=""):
    description = f"ttbar SF as a function of {var_name}"
    if not var_description:
        var_description = var_name

    weight = schemav2.Correction(
        name=f"ttbar_SF_{var_name}",
        version=1,
        description=description,
        inputs=[
            schemav2.Variable(
                name=var_name,
                type="real",
                description=var_description,
            ),
            schemav2.Variable(
                name="systematic",
                type="string",
                description="Systematic variation",
            ),
        ],
        output=schemav2.Variable(
            name="weight", type="real", description="Multiplicative event weight"
        ),
        data=schemav2.Category(
            nodetype="category",
            input="systematic",
            content=[
                {"key": "nominal", "value": binning(edges, val_nom, var_name)},
                {"key": "stat_up", "value": binning(edges, val_up, var_name)},
                {"key": "stat_dn", "value": binning(edges, val_down, var_name)},
            ],
            default=binning(edges, val_nom, var_name),
        ),
        generic_formulas=[],
    )
    cset = schemav2.CorrectionSet(
        schema_version=2,
        description=description,
        corrections=[
            weight,
        ],
        compound_corrections=[],
    )

    return cset



In [ ]:
mc_stack_order = ["ttbar", "VV+VJ", "QCD", "ttH"]
legend_label_dict = {
    "data": "Data",
    "QCD": "QCD Multijet",
    "ttbar": r"$t\bar{t}$",
    "ttH": r"$t\bar{t}H$",
    # "VJ": "V+jets",
    # "VV": "VV",
    "VV+VJ": "V+jets, VV",
}
color_dict = {
    "QCD": "#3f90da",
    "ttbar": "#832db6",
    "ttH": "#b9ac70",
    "VV+VJ": "#92dadd",
}


In [ ]:
corr_dict = {}

# Semi-leptonic

In [ ]:
dir_trees = Path("../semi-leptonic/trees/years")
tree_dict = {}
for year in ("2022", "2023"):
    tree_data = uproot.open(dir_trees / f"Histograms_{year}_data.root")["tree"]
    tree_QCD = uproot.open(dir_trees / f"Histograms_{year}_MC_QCD.root")["tree"]
    tree_ttbar = uproot.open(dir_trees / f"Histograms_{year}_MC_TTbar.root")["tree"]
    tree_ttH = uproot.open(dir_trees / f"Histograms_{year}_MC_ttHto2B.root")["tree"]
    tree_VJ = uproot.open(dir_trees / f"Histograms_{year}_MC_VJ.root")["tree"]
    tree_VV = uproot.open(dir_trees / f"Histograms_{year}_MC_VV.root")["tree"]
    tree_dict[year] = {
        "data": tree_data,
        "QCD": tree_QCD,
        "ttbar": tree_ttbar,
        "ttH": tree_ttH,
        "VJ": tree_VJ,
        "VV": tree_VV,
    }

In [ ]:
def get_semileptonic_mask(tree):
    dR_J1FJ = tree["dR_J1FJ"].array(library="numpy")
    dR_J2FJ = tree["dR_J2FJ"].array(library="numpy")
    dR_JmaxL = tree["dR_JmaxL"].array(library="numpy")
    # return ~((dR_J1FJ < 0.0 & dR_J2FJ < 0.0) | dR_JmaxL > 3.5)
    return ((dR_J1FJ >= 0.0) | (dR_J2FJ >= 0.0)) & (dR_JmaxL <= 3.5)


## fatJet1_Tau3OverTau2

In [ ]:
# fatJet1_Tau3OverTau2
bins = np.arange(0.15, 1+0.0001, 0.05)
print(f"Bins: {bins}")
feature_dict = {}
for year, trees in tree_dict.items():
    feature_dict[year] = {}
    for sample_name, tree in trees.items():
        mask = get_semileptonic_mask(tree)
        feature_dict[year][sample_name] = {
            "fatJet1_Tau3OverTau2": tree["fatJet1_Tau3OverTau2"].array(library="numpy")[mask],
            "weight": tree["weight"].array(library="numpy")[mask],
        }

In [ ]:
kfact = {}
feature_name = "fatJet1_Tau3OverTau2"
feature_label = "tau32"

for year, trees in tree_dict.items():
    
    # Initialize storage for histograms
    data_hist = None
    mc_hists = []
    
    for sample_name, tree in trees.items():
        feature = feature_dict[year][sample_name][feature_name]
        weights = feature_dict[year][sample_name]["weight"]
        
        # Calculate histogram
        counts, edges = np.histogram(feature, bins=bins, weights=weights)
        
        if sample_name == "data":
            data_hist = counts
        else:
            # Store histograms for ratio calculation
            mc_hists.append(counts)
    
    total_bkg = np.sum(mc_hists, axis=0)
    
    # Avoid division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = data_hist.sum() / total_bkg.sum()
        # ratio_err = ratio * np.sqrt(1/data_hist.sum() + 1/total_bkg.sum())
        # print(f"Year {year}: Data / Total Bkg = {ratio:.3f} ± {ratio_err    :.3f}")
        kfact[year] = ratio
        
print(f"k-factors: {kfact}")

In [ ]:
sf_dict = {}

for year, trees in tree_dict.items():
    # Create figure with subplots (2:1 ratio)
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(10, 10), 
        gridspec_kw={'height_ratios': [4, 1], 'hspace': 0.05}
    )
    
    # Initialize storage for histograms
    ttbar_hist = None
    bkg_hists = []  # MC background (MC - ttbar)
    
    # MC
    mc_hists = []
    mc_labels = []
    mc_colors = []
    for sample_name in mc_stack_order:
        if sample_name == "VV+VJ":
            feature = np.concatenate([
                feature_dict[year]["VJ"][feature_name],
                feature_dict[year]["VV"][feature_name],
            ])
            weights = np.concatenate([
                feature_dict[year]["VJ"]["weight"],
                feature_dict[year]["VV"]["weight"],
            ])
        else:
            feature = feature_dict[year][sample_name][feature_name]
            weights = feature_dict[year][sample_name]["weight"]
    
        weights = weights * kfact[year]
        
        # Calculate histogram
        counts, edges = np.histogram(feature, bins=bins, weights=weights)
        mc_hists.append(counts)
        mc_labels.append(legend_label_dict[sample_name])
        mc_colors.append(color_dict[sample_name])
        
        if sample_name == "ttbar":
            ttbar_hist = counts
        else:
            bkg_hists.append(counts)
    
    if bkg_hists:
        total_bkg = np.sum(bkg_hists, axis=0)
    else:
        total_bkg = np.zeros_like(data_hist)
        
    hep.histplot(
        mc_hists,
        bins=bins,
        stack=True,
        histtype="fill",
        color=mc_colors,
        label=mc_labels,
        ax=ax1
    )
    
    # Data
    feature = feature_dict[year]["data"][feature_name]
    weights = feature_dict[year]["data"]["weight"]
    counts, edges = np.histogram(feature, bins=bins, weights=weights)
    data_hist = counts
    data_err = np.sqrt(counts)
    hep.histplot(
        (counts, edges),
        label=legend_label_dict["data"],
        xerr=False,
        yerr=data_err,
        ax=ax1,
        color="black",
        histtype="errorbar",
    )
    
    # Reorder legend to put data first
    handles, labels = ax1.get_legend_handles_labels()
    data_handle = handles[-1]
    data_label = labels[-1]
    new_handles = [data_handle] + handles[:-1]
    new_labels = [data_label] + labels[:-1]
    leg = ax1.legend(new_handles, new_labels, bbox_to_anchor=(0, 0.82), loc="upper left")
    # for handle in leg.get_patches():
    #     handle.set_edgecolor('black')
    #     handle.set_linewidth(1.5)
        
    # Top subplot formatting
    ax1.set_ylabel("Events")
    if year in ylim_dict:
        ax1.set_ylim(ylim_dict[year])
    ax1.set_yscale("log")
    ax1.set_xticklabels([])  # Hide x-axis labels for top plot
    # ax1.grid(True, alpha=0.3)
    
    # Derive scale factors
    with np.errstate(divide='ignore', invalid='ignore'):
        sf = (data_hist - total_bkg) / ttbar_hist
        mask = (ttbar_hist == 0) | (sf < 0) | (sf > 2)
        sf[mask] = 1  # Set to 0 where TTbar is 0
        print(f"Year: {year}, Ratios: {sf}")
        
        sf_err = data_err / ttbar_hist
        sf_err[mask] = 0
        
        sf_dict[year] = {
            "SF": sf,
            "err": sf_err,
        }
        
        sf_dict[year] = {
            "SF": sf,
            "err": sf_err,
        }
    
    # Plot ratio
    bin_centers = (edges[:-1] + edges[1:]) / 2
    bin_widths = edges[1:] - edges[:-1]    
    ax2.errorbar(
        bin_centers, 
        sf, 
        yerr=sf_err, 
        xerr=False, 
        fmt='ko', 
        markersize=4, 
        capsize=0
    )
    ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
    
    # Bottom subplot formatting
    ax2.set_xlabel(r"$\tau_{3} / \tau_{2}$")
    ax2.set_ylabel(r"$t\bar{t}$ SF")
    ax2.set_ylim(0, 2)  # Adjust as needed
    # ax2.grid(True, alpha=0.3)
    
    for ax in (ax1, ax2):
        ax.set_xlim(bins[0], bins[-1])
    
    hep.cms.label(
        "Preliminary",
        fontsize=24,
        data=True,
        lumi=f"{LUMI_DICT[year] / 1e3:.0f}",
        year=None,
        ax=ax1,
        com="13",
        loc=1,
    )
    
    title = r"$t\bar{t}$ semileptonic CR"
    x_text = 0.02
    # y_text = 0.78
    y_text = 0.76
    ax1.text(x_text + 0.03, y_text + 0.06, title, fontsize=24, transform=ax1.transAxes)
    

    plt.tight_layout()
    plt.savefig(plot_dir / f"{feature_name}_{year}.pdf", bbox_inches='tight')
    plt.show()

In [ ]:
for year, sf in sf_dict.items():
    sf_nominal = sf["SF"]
    sf_err = sf["err"]
    sf_up = sf_nominal + sf_err
    sf_down = sf_nominal - sf_err
    edges = bins
    cset = get_corr(edges, sf_nominal, sf_up, sf_down, var_name="tau32", var_description="FatJet Tau3 over Tau2")
    with open(sf_dir / f"corr_{feature_label}_{year}.json", "w") as fout:
        fout.write(cset.model_dump_json(exclude_unset=True))
        
corr_dict[feature_label] = {}
for year in sf_dict:
    corr_file = sf_dir / f"corr_{feature_label}_{year}.json"
    if not corr_file.exists():
        raise FileNotFoundError(f"Correction file {corr_file} does not exist.")

    # Load the correction
    corr = correctionlib.CorrectionSet.from_file(str(corr_file))
    corr_dict[feature_label][year] = corr
    print(f"Loaded correction for {year} from {corr_file}")

In [ ]:
for year, trees in tree_dict.items():
    # Create figure with subplots (2:1 ratio)
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(10, 10), 
        gridspec_kw={'height_ratios': [4, 1], 'hspace': 0.05}
    )
    
    # Initialize storage for histograms
    ttbar_hist = None
    bkg_hists = []  # MC background (MC - ttbar)
    
    # MC
    mc_hists = []
    mc_labels = []
    mc_colors = []
    for sample_name in mc_stack_order:
        if sample_name == "VV+VJ":
            feature = np.concatenate([
                feature_dict[year]["VJ"][feature_name],
                feature_dict[year]["VV"][feature_name],
            ])
            weights = np.concatenate([
                feature_dict[year]["VJ"]["weight"],
                feature_dict[year]["VV"]["weight"],
            ])
        else:
            feature = feature_dict[year][sample_name][feature_name]
            weights = feature_dict[year][sample_name]["weight"]
    
        weights = weights * kfact[year]
        if sample_name == "ttbar":
            # Apply SF correction
            cset = corr_dict["tau32"][year]
            corr = cset["ttbar_SF_tau32"]
            sf_nom = corr.evaluate(feature, "nominal")
            sf_up = corr.evaluate(feature, "stat_up")
            sf_down = corr.evaluate(feature, "stat_dn")
            weights = weights * sf_nom
        
        # Calculate histogram
        counts, edges = np.histogram(feature, bins=bins, weights=weights)
        mc_hists.append(counts)
        mc_labels.append(legend_label_dict[sample_name])
        mc_colors.append(color_dict[sample_name])
        
        if sample_name == "ttbar":
            ttbar_hist = counts
        else:
            bkg_hists.append(counts)
    
    if bkg_hists:
        total_bkg = np.sum(bkg_hists, axis=0)
    else:
        total_bkg = np.zeros_like(data_hist)
        
    hep.histplot(
        mc_hists,
        bins=bins,
        stack=True,
        histtype="fill",
        color=mc_colors,
        label=mc_labels,
        ax=ax1
    )
    
    # Data
    feature = feature_dict[year]["data"][feature_name]
    weights = feature_dict[year]["data"]["weight"]
    counts, edges = np.histogram(feature, bins=bins, weights=weights)
    data_hist = counts
    data_err = np.sqrt(counts)
    hep.histplot(
        (counts, edges),
        label=legend_label_dict["data"],
        xerr=False,
        yerr=data_err,
        ax=ax1,
        color="black",
        histtype="errorbar",
    )
    
    # Reorder legend to put data first
    handles, labels = ax1.get_legend_handles_labels()
    data_handle = handles[-1]
    data_label = labels[-1]
    new_handles = [data_handle] + handles[:-1]
    new_labels = [data_label] + labels[:-1]
    leg = ax1.legend(new_handles, new_labels, bbox_to_anchor=(0, 0.82), loc="upper left")
    # # Patch
    # for handle in leg.get_patches():
    #     handle.set_edgecolor('black')
    #     handle.set_linewidth(1.5)
        
    # Top subplot formatting
    ax1.set_ylabel("Events")
    if year in ylim_dict:
        ax1.set_ylim(ylim_dict[year])
    ax1.set_yscale("log")
    ax1.set_xticklabels([])  # Hide x-axis labels for top plot
    # ax1.grid(True, alpha=0.3)
    
    # Derive scale factors
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = (data_hist - total_bkg) / ttbar_hist
        print(f"Year: {year}, Ratios: {ratio}")
        ratio_err = data_err / ttbar_hist
    
    # Plot ratio
    bin_centers = (edges[:-1] + edges[1:]) / 2
    bin_widths = edges[1:] - edges[:-1]    
    ax2.errorbar(
        bin_centers, 
        ratio, 
        yerr=ratio_err, 
        xerr=False, 
        fmt='ko', 
        markersize=4, 
        capsize=0
    )
    ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
    
    # Bottom subplot formatting
    ax2.set_xlabel(r"$\tau_{3} / \tau_{2}$")
    ax2.set_ylabel(r"$t\bar{t}$ SF")
    ax2.set_ylim(0, 2)
    # ax2.grid(True, alpha=0.3)
    
    for ax in (ax1, ax2):
        ax.set_xlim(bins[0], bins[-1])
    
    hep.cms.label(
        "Preliminary",
        fontsize=24,
        data=True,
        lumi=f"{LUMI_DICT[year] / 1e3:.0f}",
        year=None,
        ax=ax1,
        com="13",
        loc=1,
    )
    
    title = r"$t\bar{t}$ semileptonic CR"
    x_text = 0.02
    # y_text = 0.78
    y_text = 0.76
    ax1.text(x_text + 0.03, y_text + 0.06, title, fontsize=24, transform=ax1.transAxes)
    

    plt.tight_layout()
    plt.savefig(plot_dir / f"{feature_name}_{year}-postcorr.pdf", bbox_inches='tight')
    plt.show()